# Dataset Splitting
Create stratified 80/10/10 train, validation, and test splits for all datasets.

In [6]:
# DEV MODE keeps notebook runs small during development; set to False for full DGX experiments.
DEV_MODE = True
SAMPLE_SIZE = 10000

# Resolve the project root from either the repository root or notebooks/ directory.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the shared device and seed utilities required by the specification.
from utils.device import RANDOM_SEED, get_device, set_seed

set_seed(RANDOM_SEED)
DEVICE = get_device()


Selected device: CUDA (NVIDIA GeForce RTX 5070 Ti Laptop GPU)


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split


## Load Processed Datasets

In [8]:
processed_dir = PROJECT_ROOT / "data/processed"
datasets = {
    "drugs_com": pd.read_csv(processed_dir / "drugs_com_clean.csv"),
    "webmd": pd.read_csv(processed_dir / "webmd_clean.csv"),
    "druglib": pd.read_csv(processed_dir / "druglib_clean.csv"),
}
if DEV_MODE:
    datasets = {name: frame.head(SAMPLE_SIZE).copy() for name, frame in datasets.items()}
for name, frame in datasets.items():
    print(name, frame.shape, frame["label"].value_counts(normalize=True).sort_index().to_dict())


drugs_com (9788, 3) {0: 0.24979566816510013, 1: 0.09194932570494484, 2: 0.658255006129955}
webmd (7649, 11) {0: 0.43352072166296246, 1: 0.14511700875931494, 2: 0.4213622695777226}
druglib (1084, 12) {0: 0.48985239852398527, 1: 0.17435424354243542, 2: 0.33579335793357934}


## Stratified Split Helper

In [9]:
def stratified_80_10_10(frame):
    # Split a dataframe with stratification on the sentiment label.
    train_df, temp_df = train_test_split(
        frame,
        test_size=0.20,
        random_state=RANDOM_SEED,
        stratify=frame["label"],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=RANDOM_SEED,
        stratify=temp_df["label"],
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


split_dir = PROJECT_ROOT / "data/splits"
split_dir.mkdir(parents=True, exist_ok=True)
saved_paths = []
for name, frame in datasets.items():
    train_df, val_df, test_df = stratified_80_10_10(frame)
    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        path = split_dir / f"{name}_{split_name}.csv"
        split_df.to_csv(path, index=False)
        saved_paths.append(path)
        print(path.name, split_df.shape, split_df["label"].value_counts(normalize=True).sort_index().to_dict())


drugs_com_train.csv (7830, 3) {0: 0.24980842911877393, 1: 0.09195402298850575, 2: 0.6582375478927203}
drugs_com_val.csv (979, 3) {0: 0.24923391215526047, 1: 0.09193054136874361, 2: 0.658835546475996}
drugs_com_test.csv (979, 3) {0: 0.25025536261491316, 1: 0.09193054136874361, 2: 0.6578140960163432}
webmd_train.csv (6119, 11) {0: 0.4335675764013728, 1: 0.1451217519202484, 2: 0.4213106716783788}
webmd_val.csv (765, 11) {0: 0.4326797385620915, 1: 0.1450980392156863, 2: 0.4222222222222222}
webmd_test.csv (765, 11) {0: 0.4339869281045752, 1: 0.1450980392156863, 2: 0.42091503267973857}
druglib_train.csv (867, 12) {0: 0.49019607843137253, 1: 0.17416378316032297, 2: 0.3356401384083045}
druglib_val.csv (108, 12) {0: 0.49074074074074076, 1: 0.17592592592592593, 2: 0.3333333333333333}
druglib_test.csv (109, 12) {0: 0.48623853211009177, 1: 0.1743119266055046, 2: 0.3394495412844037}


## Reload Verification

In [10]:
for path in saved_paths:
    print(path.name, pd.read_csv(path).shape)


drugs_com_train.csv (7830, 3)
drugs_com_val.csv (979, 3)
drugs_com_test.csv (979, 3)
webmd_train.csv (6119, 11)
webmd_val.csv (765, 11)
webmd_test.csv (765, 11)
druglib_train.csv (867, 12)
druglib_val.csv (108, 12)
druglib_test.csv (109, 12)
